In [1]:
import os
if not os.path.exists("Food-Freshness-Image-Prediction-System"):
    os.system("git clone https://github.com/nickymarzz/Food-Freshness-Image-Prediction-System.git")
os.chdir("Food-Freshness-Image-Prediction-System")


Cloning into 'Food-Freshness-Image-Prediction-System'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 114 (delta 39), reused 111 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 15.31 MiB | 16.76 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Food-Freshness-Image-Prediction-System


In [2]:
import os
os.system("pip install -q -r requirements.txt")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 29.0 MB/s eta 0:00:00


In [3]:
import tensorflow as tf
print("GPU available:", tf.config.list_physical_devices('GPU'))

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
import os
os.system("pip install -q opendatasets")


In [10]:
# Before importing opendatasets, create a mock 'cgi' module for Python 3.13+ compatibility
import sys
if 'cgi' not in sys.modules:
    try:
        from wsgiref import headers

        class MockCGIModule:
            def parse_header(self, value):
                return headers.parse_header(value)

            def parse_multipart(self, fp, pdata, *args, **kwargs):
                # This function is more complex to mock and often not directly replaceable.
                # If opendatasets requires this, it might still fail.
                raise NotImplementedError("cgi.parse_multipart is not available in Python 3.13 and cannot be easily mocked. opendatasets might be incompatible for this dataset.")

        sys.modules['cgi'] = MockCGIModule()
    except ImportError:
        print("Warning: Could not create a mock cgi module as wsgiref is missing.")

import opendatasets as od
od.download("https://www.kaggle.com/datasets/muhriddinmuxiddinov/fruits-and-vegetables-dataset")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: nickymarzz
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/muhriddinmuxiddinov/fruits-and-vegetables-dataset


100%|██████████| 870M/870M [00:14<00:00, 61.9MB/s]


In [20]:
import os
import shutil
from pathlib import Path

# Source raw data path
raw_path = Path("artifacts/data/raw")
kaggle_path = Path("fruits-and-vegetables-dataset")

# Define produce categories and item names
categories = {
    'Fruits': ['Mango', 'Banana', 'Orange', 'Apple', 'Strawberry'],
    'Vegetables': ['Cucumber', 'Bellpepper', 'Tomato', 'Potato', 'Carrot']
}

# Organize images into: artifacts/data/raw/{Fruits|Vegetables}/{Item}/{Fresh|Rotten}/
for cat_name, items in categories.items():
    cat_dir = raw_path / cat_name
    os.makedirs(cat_dir, exist_ok=True)

    for item in items:
        item_dir = cat_dir / item
        os.makedirs(item_dir, exist_ok=True)

        for state in ['Fresh', 'Rotten']:
            dst = item_dir / state
            os.makedirs(dst, exist_ok=True)

            # Check source from Kaggle download or any temporary folder
            candidate_sources = [
                kaggle_path / cat_name / f"{state}{item}",
                raw_path / state / f"{state}{item}",
                cat_dir / f"{state}{item}"
            ]

            for src in candidate_sources:
                if src.exists():
                    for file in src.glob("*.*"):
                        target_file = dst / file.name
                        if not target_file.exists():
                            shutil.move(str(file), str(target_file))
                    try:
                        src.rmdir()
                    except OSError:
                        pass

# Clean up temporary folders if present
for temp_folder in [raw_path / "Fresh", raw_path / "Rotten"]:
    if temp_folder.exists():
        try:
            shutil.rmtree(temp_folder)
        except Exception:
            pass

# Verify structure
print("Contents of artifacts/data/raw:", os.listdir(raw_path))
for cat in os.listdir(raw_path):
    cat_p = raw_path / cat
    if cat_p.is_dir():
        sub = os.listdir(cat_p)[:2]
        print(f"  Category '{cat}' subfolders:", sub)
        if sub:
            item_p = cat_p / sub[0]
            if item_p.is_dir():
                print(f"    Contents of '{sub[0]}':", os.listdir(item_p))


Contents of artifacts/data/raw: ['Fruits', 'Vegetables']
  Category 'Fruits' subfolders: ['Mango', 'Banana']
    Contents of 'Mango': ['Fresh', 'Rotten']
  Category 'Vegetables' subfolders: ['Carrot', 'Bellpepper']
    Contents of 'Carrot': ['Fresh', 'Rotten']


In [21]:
import os
os.system("python -m src.pipeline.training_pipeline")


STEP 1 – Data Ingestion
[2026-09-30 13:50:33,147] INFO: ============================================================
[2026-09-30 13:50:33,147] INFO: STARTING DATA INGESTION PIPELINE
[2026-09-30 13:50:33,147] INFO: ============================================================
[2026-09-30 13:50:33,147] INFO: Starting data validation
[2026-09-30 13:50:33,298] INFO: Class 'Fresh' contains 6108 images
[2026-09-30 13:50:33,298] INFO: Class 'Rotten' contains 5878 images
[2026-09-30 13:50:33,298] INFO: Validation report saved to /content/Food-Freshness-Image-Prediction-System/artifacts/data/validation_report.json
[2026-09-30 13:50:33,298] INFO: Data validation PASSED. Total images: 11986
[2026-09-30 13:50:33,307] INFO: Checking image file integrity
[2026-09-30 13:50:35,681] INFO: All image files passed integrity check
[2026-09-30 13:50:35,693] INFO: Generating dataset metadata
[2026-09-30 13:50:35,905] INFO: Metadata saved to /content/Food-Freshness-Image-Prediction-System/artifacts/data/metada

In [22]:
from google.colab import files
import shutil

# Zip the models directory
shutil.make_archive("retrained_models", "zip", "artifacts/models")

# Download to your computer
files.download("retrained_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import os
os.system("python -m src.components.model_evaluation")


2026-09-30 14:22:38.427125: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790778158.428619   26795 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1800 files belonging to 2 classes.
2026-09-30 14:22:51.712246: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-30 14:22:51.850211: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1790778173.588125   26834 device_compile

In [28]:
import os
os.system("python -m src.components.category_model_evaluation")


2026-09-30 14:23:57.182393: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1790778237.183921   27327 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Found 1789 files belonging to 10 classes.
2026-09-30 14:24:10.780766: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-30 14:24:10.917848: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1790778252.521921   27367 device_compil

In [29]:
from google.colab import files
import shutil
shutil.make_archive("evaluation_results", "zip", "artifacts/results")
files.download("evaluation_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>